In [1]:
import pandas as pd

# 1. Load the dataset
df = pd.read_csv('data/resume_data.csv')

# --- THE BUG FIX ---
# Scrub invisible ghost characters (BOM) and spaces off the column headers
df.columns = [col.replace('\ufeff', '').strip() for col in df.columns]

# 2. Select only the columns we care about
df = df[['skills', 'job_position_name']].dropna()
df.columns = ['Skills', 'Role']

# 3. Clean the text
# Right now, skills look like "['Python', 'Java']"
# We want to strip the brackets and quotes so it's just pure text: "Python Java"
df['Skills'] = df['Skills'].str.replace(r"[\[\]\']", "", regex=True).str.replace(",", " ")

# 4. Let's see how many roles we have
print("Total Resumes:", len(df))
print("\nTop 5 Job Roles in dataset:")
print(df['Role'].value_counts().head(5))

# 5. Show the cleaned data
df.head()

Total Resumes: 9488

Top 5 Job Roles in dataset:
Role
Project Coordinator (Civil)    340
HR Officer                     340
Civil Engineer                 340
Site Engineer                  340
Senior Software Engineer       339
Name: count, dtype: int64


,Skills,Role
0,Big Data Hadoop Hive Python Mapreduce Spa...,Senior Software Engineer
1,Data Analysis Data Analytics Business Analys...,Machine Learning (ML) Engineer
2,Software Development Machine Learning Deep L...,"Executive/ Senior Executive- Trade Marketing, ..."
3,accounts payables accounts receivables Accou...,Business Development Executive
4,Analytical reasoning Compliance testing knowl...,Senior iOS Engineer


In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# 1. Our Clean, Hand-Crafted Dataset
clean_data = {
    "Skills": [
        "Python Django RESTAPI SQL Postgres Backend",
        "React NextJS CSS HTML Tailwind Frontend",
        "Python Scikit-Learn Pandas Machine Learning AI",
        "Figma Sketch UI UX Wireframing Design",
        "Python FastAPI Docker AWS Kubernetes Backend",
        "JavaScript React Redux CSS Frontend",
        "TensorFlow PyTorch Deep Learning NLP AI",
        "Adobe XD Figma Prototyping User Testing Design",
        "Java Spring Boot SQL Microservices Backend",
        "VueJS NuxtJS HTML CSS Frontend"
    ],
    "Role": [
        "Backend Developer", "Frontend Developer", "Data Scientist", "UI/UX Designer",
        "Backend Developer", "Frontend Developer", "Data Scientist", "UI/UX Designer",
        "Backend Developer", "Frontend Developer"
    ]
}

df = pd.DataFrame(clean_data)

# 2. Split the data
X_train, X_test, y_train, y_test = train_test_split(df['Skills'], df['Role'], test_size=0.2, random_state=42)

# 3. Convert words to Math (TF-IDF)
vectorizer = TfidfVectorizer() 
X_train_vectors = vectorizer.fit_transform(X_train)
X_test_vectors = vectorizer.transform(X_test)

# 4. Train the Model
model = LogisticRegression()
model.fit(X_train_vectors, y_train)

# 5. Test the Model
y_pred = model.predict(X_test_vectors)
accuracy = accuracy_score(y_test, y_pred)
print(f"✅ Model Accuracy on Clean Data: {accuracy * 100:.2f}%\n")

# --- TEST YOUR OWN SKILLS ---
# Let's see what the AI predicts for you now!
my_skills = ["Python SQL React NextJS FastAPI Machine Learning AI Data Science"]

my_skills_math = vectorizer.transform(my_skills)
predicted_role = model.predict(my_skills_math)
probabilities = model.predict_proba(my_skills_math)[0] # Get the confidence scores

print(f"🎯 The AI predicts your Job Role is: {predicted_role[0]}")
print("\n📊 Confidence Breakdown:")
for role, prob in zip(model.classes_, probabilities):
    print(f" - {role}: {prob * 100:.2f}%")

✅ Model Accuracy on Clean Data: 100.00%

🎯 The AI predicts your Job Role is: Data Scientist

📊 Confidence Breakdown:
 - Backend Developer: 27.70%
 - Data Scientist: 32.00%
 - Frontend Developer: 21.42%
 - UI/UX Designer: 18.89%
